# Storing pipeline outputs: Delta Lake vs. catcif

This notebook walks through the two storage backends for `PipelineDataHandler.store()`:

1. **`format="delta"`** (the existing default) — writes a Delta Lake table under `store_path/`. Each structure becomes a row whose `content` column holds the CIF bytes, addressed by its original file path. Portable only via the `deltalake` / `polars` stack, but supports columnar scans.
2. **`format="catcif"`** (new) — writes a single `.catcif` archive: concatenated-CIF. Each structure becomes one archive entry, and the handler's paths are rewritten to self-describing `archive.catcif:entry_id` tags.

Both backends **preserve row count** — they only consolidate on-disk file layout.

We use a tiny fixture structure (`tests/test_data/unittests/rfd3/L1.cif`) to simulate a 5-design pipeline across two steps (`rfd3` -> `mpnn`) without needing a GPU.

In [ ]:
import shutil
import tempfile
from pathlib import Path

import polars as pl

from pipelines.data import PipelineDataHandler
from pipelines.data.schema import Column, StructureColumn, make_row
from pipelines.data.catcif import list_catcif_entries

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
FIXTURE = REPO / "tests" / "test_data" / "unittests" / "rfd3" / "L1.cif"

WORK = Path(tempfile.mkdtemp(prefix="catcif_demo_"))
print("scratch dir:", WORK)

## 1. Build a tiny two-step handler

We fabricate 5 `rfd3` structures and 5 `mpnn` structures (one per row). Each structure is just a copy of the fixture CIF — enough to exercise the storage APIs end-to-end.

In [ ]:
def make_fake_step(step: str, n: int) -> list[Path]:
    d = WORK / step
    d.mkdir(parents=True, exist_ok=True)
    content = FIXTURE.read_text()
    paths = []
    for i in range(n):
        p = d / f"{step}_{i:04d}.cif"
        p.write_text(content)
        paths.append(p)
    return paths


rfd3_paths = make_fake_step("rfd3", 5)
mpnn_paths = make_fake_step("mpnn", 5)

rows = []
for i, (rfd3, mpnn) in enumerate(zip(rfd3_paths, mpnn_paths)):
    row = make_row("rfd3", str(rfd3), metrics={"rfd3.plddt": 80 + i})
    row[Column.STRUCTURES].append(
        {
            StructureColumn.NAME: "mpnn",
            StructureColumn.PATH: str(mpnn),
            StructureColumn.PATH_HANDLE: None,
            StructureColumn.STORE_PATH: None,
            StructureColumn.STORAGE_FORMAT: None,
            StructureColumn.METADATA: None,
        }
    )
    rows.append(row)

handler = PipelineDataHandler(rows)
print(f"rows = {len(handler)}, steps = {handler.names}")
handler.unpack()

## 2. Store in Delta Lake (current default)

`handler.store(step, store_path)` or equivalently `handler.store(step, store_path, format="delta")`.

The store is a directory containing parquet segments and a `_delta_log/` directory. Structure content is CIF bytes keyed by `original_path`.

In [ ]:
delta_handler = (
    PipelineDataHandler(rows)
    .store("rfd3", WORK / "delta_store" / "rfd3", format="delta")
    .store("mpnn", WORK / "delta_store" / "mpnn", format="delta")
)

for p in sorted((WORK / "delta_store").rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(WORK)):<60} {p.stat().st_size:>8d} B")

In [ ]:
# Peek inside the Delta table
delta = pl.read_delta(str(WORK / "delta_store" / "rfd3"))
print("Delta schema:", delta.schema)
delta.select(
    [
        "original_path",
        "materialized_name",
        pl.col("content").bin.size().alias("content_bytes"),
    ]
)

In [ ]:
# The handler retains original paths; store_path/storage_format are the retrieval hooks
delta_handler.unpack()[
    [
        "rfd3.path",
        "rfd3.store_path",
        "rfd3.storage_format",
        "mpnn.path",
        "mpnn.store_path",
        "mpnn.storage_format",
    ]
]

## 3. Store in catcif

`handler.store(step, store_path, format="catcif", cleanup=False)`.

- `store_path` names a single file. The `.catcif` suffix is added automatically.
- The handler's `rfd3.path` becomes `rfd3.catcif:rfd3_0000` — self-describing.
- `cleanup=True` deletes the original files after the archive is written (opt-in).

In [ ]:
catcif_handler = (
    PipelineDataHandler(rows)
    .store(
        "rfd3", WORK / "catcif_store" / "rfd3.catcif", format="catcif", cleanup=False
    )
    .store(
        "mpnn", WORK / "catcif_store" / "mpnn.catcif", format="catcif", cleanup=False
    )
)

for p in sorted((WORK / "catcif_store").rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(WORK)):<60} {p.stat().st_size:>8d} B")

In [ ]:
print(
    "rfd3.catcif entries:", list_catcif_entries(WORK / "catcif_store" / "rfd3.catcif")
)
print(
    "mpnn.catcif entries:", list_catcif_entries(WORK / "catcif_store" / "mpnn.catcif")
)

catcif_handler.unpack()[
    ["rfd3.path", "rfd3.storage_format", "mpnn.path", "mpnn.storage_format"]
]

### 3a. Row count is preserved

This is the contract: storage only rewrites paths — it never splits, merges, or filters rows.

In [ ]:
assert len(catcif_handler) == len(handler) == 5
assert len(delta_handler) == 5
print("OK: all three handlers have 5 rows")

### 3b. Retrieval is transparent

A catcif tag is a valid argument to `parse_to_atom_array` — no manual unpack needed.

In [ ]:
from pipelines.io import parse_to_atom_array

# Grab the first rfd3 tag
row0 = next(catcif_handler._df.iter_rows(named=True))
rfd3_tag = next(
    s for s in row0[Column.STRUCTURES] if s[StructureColumn.NAME] == "rfd3"
)[StructureColumn.PATH]
print("loading:", rfd3_tag)
aa = parse_to_atom_array(rfd3_tag)
print(f"AtomArray: {len(aa)} atoms")

## 4. `materialize()`: extract back to files

By default, `materialize(output_dir)` extracts every stored entry to an individual `.cif` file, regardless of backend.

For catcif specifically, you can pass `expand_catcif=False` to keep the compact tag form and just copy the archive reference — useful when a downstream block can consume catcif tags directly.

In [ ]:
# Expand both backends into individual files
expanded = catcif_handler.materialize(WORK / "expanded", expand_catcif=True)
for p in sorted((WORK / "expanded").rglob("*.cif")):
    print(p.relative_to(WORK))

In [ ]:
# Keep catcif tags (no extraction) — downstream code can still parse them.
catcif_handler2 = PipelineDataHandler(rows).store(
    "rfd3", WORK / "catcif_store2" / "rfd3.catcif", format="catcif"
)
kept = catcif_handler2.materialize(WORK / "kept", expand_catcif=False)

paths = [
    s[StructureColumn.PATH]
    for row in kept._df.iter_rows(named=True)
    for s in row[Column.STRUCTURES]
    if s[StructureColumn.NAME] == "rfd3"
]
print("\n".join(paths))

## 5. Cleanup behaviour

`cleanup=True` on the catcif backend deletes the original files once they've been archived. This is **opt-in** — the default is to leave the working directory intact so you can rerun the pipeline without re-running upstream blocks.

In [ ]:
# Recreate files so we don't interfere with earlier cells
rfd3_paths2 = make_fake_step("rfd3_cleanup", 3)
handler_cleanup = PipelineDataHandler(
    [make_row("rfd3_cleanup", str(p)) for p in rfd3_paths2]
)
before = [p.exists() for p in rfd3_paths2]
handler_cleanup.store(
    "rfd3_cleanup",
    WORK / "cleanup_store" / "rfd3.catcif",
    format="catcif",
    cleanup=True,
)
after = [p.exists() for p in rfd3_paths2]
print(f"before cleanup: {before}")
print(f"after  cleanup: {after}")

## 6. Using it from a block

In a pipeline script you just set `store="catcif"` on the block (or `store="delta"` for the Delta backend):

```python
blocks = [
    RFD3(n_batches=50, store="catcif", store_cleanup=True),  # compact sandbox
    MPNN(store=True),                                         # delta (== legacy)
    AF3(store="catcif"),
]
```



In [ ]:
# Tidy up
shutil.rmtree(WORK, ignore_errors=True)
print(f"removed {WORK}")